# Jira Issue Writer — Gemma 4 (E4B) + Unsloth + QLoRA

Veri seti: [`fport/jira-issue-writer-tr-en`](https://huggingface.co/datasets/fport/jira-issue-writer-tr-en)
— 13.000 örnek, %50 Türkçe / %50 İngilizce, çıktı her zaman geçerli JSON.

### Önceki denemeden farklar

| | Eski notebook | Bu notebook |
|---|---|---|
| Eğitim uzunluğu | `max_steps=60` → 480 örnek (**%1**) | `num_train_epochs=2` → ~21.900 örnek |
| LoRA kapasitesi | `r=16, alpha=16` | `r=32, alpha=64` |
| Değerlendirme | yok | eval split + test setinde metrik |
| Maske kontrolü | yok | eğitimden önce doğrulanıyor |
| Öğrenme oranı | `2e-4` sabit | `1e-4` + cosine + warmup |
| Veri | genel talimat seti | göreve özel, yapılandırılmış çıktı |

**60 adım bir demoydu, eğitim değil.** Başarısızlığın ana sebebi buydu.

### Süre (Colab Pro)

| GPU | Model | 2 epoch |
|---|---|---|
| L4 24GB | gemma-4-E4B | ~2–2,5 saat |
| A100 40GB | gemma-4-E4B | ~1–1,5 saat |
| A100 40GB | gemma-4-31B (QLoRA) | ~6–8 saat |


## 1 — GPU


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
import torch
assert torch.cuda.is_available(), 'GPU yok: Runtime > Change runtime type > GPU'
VRAM = torch.cuda.get_device_properties(0).total_memory / 1024**3
GPU  = torch.cuda.get_device_name(0)
print(f'{GPU} · {VRAM:.0f} GB · bf16 {torch.cuda.is_bf16_supported()}')


## 2 — Unsloth kurulumu

Kurulumdan sonra Colab "restart session" derse **restart etme**, devam et.
Sürüm çakışması alırsan en alttaki *Sorun giderme* bölümünde pinli alternatif var.


In [ ]:
%%capture
import os
if 'COLAB_' not in ''.join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install --upgrade --no-cache-dir unsloth unsloth_zoo
    !pip install -q sentencepiece protobuf hf_transfer 'huggingface_hub>=0.34'


In [ ]:
import unsloth, transformers, trl, torch
print('unsloth', unsloth.__version__, '| transformers', transformers.__version__,
      '| trl', trl.__version__, '| torch', torch.__version__)


## 3 — Hugging Face girişi

Gemma lisansını [model sayfasından](https://huggingface.co/google/gemma-4-E4B-it) bir kez
kabul etmen gerekiyor. Token'ı Colab'ın *Secrets* bölümüne `HF_TOKEN` olarak koy.


In [ ]:
from google.colab import userdata
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import whoami
print('giris:', whoami()['name'])


## 4 — Model

`E4B` = etkin 4,5B parametre (toplam 8B, Per-Layer Embedding mimarisi), 128K bağlam.
VRAM 35GB üzerindeyse notebook 31B'ye çıkar; istemiyorsan `MODEL` satırını elle yaz.


In [ ]:
from unsloth import FastModel

MODEL   = 'unsloth/gemma-4-E4B-it'      # 10GB VRAM yeter
# MODEL = 'unsloth/gemma-4-31B-it'     # A100 40GB, QLoRA ~22GB — cok daha yavas

MAXLEN  = 2048
if VRAM >= 35:   BATCH, ACCUM = 4, 4
elif VRAM >= 22: BATCH, ACCUM = 2, 8
else:            BATCH, ACCUM = 1, 16

model, tokenizer = FastModel.from_pretrained(
    model_name     = MODEL,
    max_seq_length = MAXLEN,
    load_in_4bit   = True,
    load_in_8bit   = False,
    full_finetuning= False,
    token          = os.environ['HF_TOKEN'],
)
print(model.config.model_type, '| efektif batch', BATCH*ACCUM)


## 5 — LoRA

`r=32, alpha=64`. Eski denemedeki `r=16, alpha=16` bu iş için dar: model sadece
üslup değil, **sabit bir JSON şeması** öğrenmek zorunda. `alpha = 2r` yaygın orandır.


In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,   # metin egitiyoruz
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r            = 32,
    lora_alpha   = 64,
    lora_dropout = 0,
    bias         = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 3407,
    use_rslora   = False,
)
model.print_trainable_parameters()


## 6 — Veri


In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

tokenizer = get_chat_template(tokenizer, chat_template='gemma-4')

ds = load_dataset('fport/jira-issue-writer-tr-en')

def to_text(r):
    t = tokenizer.apply_chat_template(r['messages'], tokenize=False,
                                      add_generation_prompt=False)
    # SFTTrainer tokenize ederken BOS'u kendisi ekler; sablonun bastaki <bos>'u
    # kalirsa dizide CIFT BOS olur ve egitim bozulur (Unsloth quickstart'ta da
    # ayni removeprefix var).
    return {'text': t.removeprefix('<bos>')}

ds = ds.map(to_text, remove_columns=['messages', 'meta'], num_proc=2)
print(ds)
print('\n--- bir ornegin ham hali (ilk 900 karakter) ---')
print(ds['train'][0]['text'][:900])


### Uzunluk kontrolü

`MAXLEN`'i aşan örnek kırpılır ve kırpılan JSON kapanmaz — model bozuk çıktı üretmeyi
öğrenir. Oran %2'yi geçiyorsa `MAXLEN`'i büyüt.


In [ ]:
import random
idx = random.Random(0).sample(range(len(ds['train'])), 500)
lens = sorted(len(tokenizer(ds['train'][i]['text'])['input_ids']) for i in idx)
over = sum(l > MAXLEN for l in lens) / len(lens)
print(f'medyan {lens[len(lens)//2]} · p95 {lens[int(len(lens)*.95)]} · max {lens[-1]}')
print(f'MAXLEN={MAXLEN} asan oran: %{over*100:.1f}')
assert over < 0.05, 'cok fazla kirpma var, MAXLEN artir'

# BOS kontrolu: dizinin basinda tam olarak BIR tane BOS olmali
toks = tokenizer(ds['train'][0]['text'])['input_ids']
bos  = tokenizer.bos_token_id
n_bos_start = 0
for t in toks:
    if t == bos: n_bos_start += 1
    else: break
print(f'basta {n_bos_start} adet BOS ({tokenizer.bos_token})')
assert n_bos_start <= 1, 'CIFT BOS: to_text icindeki removeprefix calismamis'
print('BOS tamam ✓')


## 7 — Eğitim öncesi çıktı (karşılaştırma için)

Aynı promptu eğitimden sonra tekrar soracağız.


In [ ]:
from transformers import TextStreamer

SYSTEM = ('Kıdemli bir çevik teslimat asistanısın. Ham ürün girdisini düzgün yazılmış '
          'Jira kayıtlarına çevirirsin. Yalnızca tek bir geçerli JSON nesnesi döndür, '
          'başka hiçbir şey yazma. INVEST ilkelerine uy, test edilebilir Given/When/Then '
          'kabul kriterleri yaz ve asla bilgi uydurma: girdide olmayan her şey '
          '`assumptions` ya da `clarifying_questions` alanına gider.')

USER = '''Bunu bir Jira kaydına çevir.

---
selam ekip, canlı bahis tarafında şikayet var: oran değişince kupon reddediliyor,
kullanıcı ne olduğunu anlamıyor. oran yükseldiyse otomatik kabul edilsin isteyenler
var. bu sprint bakabilir miyiz?
---'''

def ask(msgs, max_new=1200, stream=True):
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True,
                                        return_tensors='pt').to('cuda')
    kw = dict(max_new_tokens=max_new, do_sample=False)
    if stream: kw['streamer'] = TextStreamer(tokenizer, skip_prompt=True)
    with torch.no_grad():
        out = model.generate(input_ids=ids, **kw)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

MSGS = [{'role':'system','content':SYSTEM},{'role':'user','content':USER}]
before = ask(MSGS)


## 8 — Trainer

`max_steps` **yok** — tam epoch üzerinden gidiyoruz. `eval_steps` ile aşırı öğrenmeyi
izleyeceğiz: eval loss düşmeyi bırakıp yükselmeye başlarsa o noktadan sonrası ezber.


In [ ]:
from trl import SFTTrainer, SFTConfig

EPOCHS = 2.0
CKPT   = '/content/drive/MyDrive/jira-gemma4-ckpt'

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = ds['train'],
    eval_dataset  = ds['validation'].select(range(256)),
    args = SFTConfig(
        dataset_text_field          = 'text',
        max_seq_length              = MAXLEN,
        per_device_train_batch_size = BATCH,
        gradient_accumulation_steps = ACCUM,
        per_device_eval_batch_size  = BATCH,
        num_train_epochs = EPOCHS,
        learning_rate    = 1e-4,          # uzun egitimde 2e-4 fazla agresif
        lr_scheduler_type= 'cosine',
        warmup_ratio     = 0.03,
        optim            = 'adamw_8bit',
        weight_decay     = 0.01,
        logging_steps    = 20,
        eval_strategy    = 'steps', eval_steps = 200,
        save_strategy    = 'steps', save_steps = 200, save_total_limit = 2,
        output_dir       = CKPT,
        seed             = 3407,
        report_to        = 'none',
    ),
)
steps = int(len(ds['train']) * EPOCHS / (BATCH*ACCUM))
print(f'{len(ds["train"])} ornek · {EPOCHS} epoch · yaklasik {steps} adim')


### Yalnızca cevaba loss uygula

Bu olmadan model senin **girdilerini de ezberler**. Gemma 4'ün tur işaretleri kullanılıyor.


In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = '<|turn>user\n',
    response_part    = '<|turn>model\n',
)
print('maskeleme uygulandi')


### Maske doğrulaması — bu hücreyi atlama

Eski notebook'ta yoktu. `instruction_part` / `response_part` metinleri modelin şablonuyla
birebir eşleşmezse maske sessizce boşa düşer ve saatlerce yanlış hedefe eğitim yaparsın.

Aşağıda **loss'a giren tokenlar** yazdırılıyor. Yalnızca JSON cevabını görmelisin;
sistem promptu veya kullanıcı mesajı görünüyorsa dur ve işaretleri düzelt.


In [ ]:
ex = trainer.train_dataset[0]
ids    = ex['input_ids']
labels = ex['labels']

kept    = [i for i, l in zip(ids, labels) if l != -100]
ignored = [i for i, l in zip(ids, labels) if l == -100]

print(f'toplam {len(ids)} token · loss\'a giren {len(kept)} (%{len(kept)/len(ids)*100:.0f})')
print('\n=== LOSS HESAPLANAN KISIM (ilk 600 karakter) ===')
print(tokenizer.decode(kept)[:600])
print('\n=== MASKELENEN KISIM (ilk 400 karakter) ===')
print(tokenizer.decode(ignored)[:400])

assert 0.15 < len(kept)/len(ids) < 0.95, (
    'Maske oraninda anormallik var. Cok dusukse isaretler eslesmiyor, '
    'cok yuksekse maskeleme hic uygulanmamis demektir.')
print('\nmaske makul görünüyor ✓')


## 9 — Eğitim

Checkpoint'ler Drive'a yazılır; oturum koparsa bu iki hücreyi tekrar çalıştır.

> **Loss hakkında:** Unsloth dokümanı E2B/E4B varyantlarında 13–15 bandındaki loss'un
> beklenen olduğunu belirtiyor (26B/31B'de 1–3). Yani **mutlak değere değil, eğilime**
> bak: `eval_loss` düşüyor mu? Asıl karar Adım 11'deki test metrikleriyle verilir.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
resume = os.path.isdir(CKPT) and any(d.startswith('checkpoint-') for d in os.listdir(CKPT))
print('kaldigi yerden devam' if resume else 'sifirdan basliyor')


In [ ]:
stats = trainer.train(resume_from_checkpoint=resume)
print(stats.metrics)


## 10 — Eğitim sonrası aynı soru


In [ ]:
import json
after = ask(MSGS, stream=False)
print('=== EGITIM SONRASI ===')
try:
    o = json.loads(after)
    print('JSON gecerli ✓')
    print('type    :', o.get('issue_type'))
    print('summary :', o.get('summary'))
    print('priority:', o.get('priority'), '| points:', o.get('story_points'))
    print('AC      :', len(o.get('acceptance_criteria', [])))
    print('varsayim:', o.get('assumptions'))
    print('sorular :', o.get('clarifying_questions'))
    print('\n' + o.get('description','')[:900])
except json.JSONDecodeError as e:
    print('JSON PARSE HATASI:', e); print(after[:1200])


## 11 — Test setinde ölçüm

Test seti eğitimde **hiç görülmemiş** içerik çekirdeklerinden gelir; ezberi değil
genellemeyi ölçer. 120 örnek E4B'de ~8 dakika.

Hedef bantlar: `json_valid` > %95, `type_acc` > %85, `no_hallucination` > %90.


In [ ]:
import json, re, collections
from datasets import load_dataset

raw  = load_dataset('fport/jira-issue-writer-tr-en', split='test')
rows = [r for r in raw if r['meta']['task'] in ('draft_issue','bug_from_log')][:120]
VER  = re.compile(r'\b\d+\.\d+(?:\.\d+)?\b')
agg  = collections.defaultdict(list)
by_lang = collections.defaultdict(lambda: collections.defaultdict(list))

for n, r in enumerate(rows, 1):
    pred = ask(r['messages'][:2], max_new=1400, stream=False)
    gold = json.loads(r['messages'][2]['content'])
    L    = r['meta']['lang']
    def put(k, v):
        agg[k].append(v); by_lang[L][k].append(v)
    try:
        p = json.loads(pred)
    except json.JSONDecodeError:
        put('json_valid', 0); continue
    put('json_valid', 1)
    put('type_acc',     int(p.get('issue_type') == gold.get('issue_type')))
    put('priority_acc', int(p.get('priority')   == gold.get('priority')))
    s = p.get('summary','')
    put('summary_ok', int(0 < len(s) <= 120 and not re.match(
        r'^\s*\[?(bug|story|task|epic)\]?\s*[:-]', s, re.I)))
    put('has_fields', int(all(k in p for k in
        ('issue_type','summary','description','priority','labels','components'))))
    if p.get('issue_type') == 'Story':
        put('ac_count_ok', int(3 <= len(p.get('acceptance_criteria') or []) <= 7))
    put('no_hallucination', int(
        set(VER.findall(json.dumps(p, ensure_ascii=False)))
        <= set(VER.findall(r['messages'][1]['content']))))
    if n % 20 == 0: print(f'  {n}/{len(rows)}')

print(f'\n{len(rows)} ornek\n')
print(f"{'metrik':18}{'tumu':>9}{'en':>9}{'tr':>9}")
for k in sorted(agg):
    def pct(d):
        v = d.get(k, [])
        return f'%{sum(v)/len(v)*100:5.1f}' if v else '    - '
    print(f'{k:18}{pct(agg):>9}{pct(by_lang["en"]):>9}{pct(by_lang["tr"]):>9}')


## 12 — Kaydet


In [ ]:
OUT = '/content/jira-writer-gemma4'
model.save_pretrained(OUT); tokenizer.save_pretrained(OUT)
import shutil
shutil.copytree(OUT, '/content/drive/MyDrive/jira-writer-gemma4', dirs_exist_ok=True)
print('Drive: MyDrive/jira-writer-gemma4')


In [ ]:
# LoRA adapter'ini Hub'a yukle
model.push_to_hub('fport/jira-writer-gemma4-lora', token=os.environ['HF_TOKEN'])
tokenizer.push_to_hub('fport/jira-writer-gemma4-lora', token=os.environ['HF_TOKEN'])


In [ ]:
# 16-bit birlestirilmis model (vLLM / dogrudan kullanim icin)
# model.push_to_hub_merged('fport/jira-writer-gemma4', tokenizer,
#                          save_method='merged_16bit', token=os.environ['HF_TOKEN'])

# GGUF (Ollama / llama.cpp icin) — donusum ~10-15 dk
# model.push_to_hub_gguf('fport/jira-writer-gemma4-gguf', tokenizer,
#                        quantization_method=['q4_k_m'], token=os.environ['HF_TOKEN'])


---

## Sorun giderme

**Maske doğrulaması patlıyor.** Gemma 4'ün tur işaretleri sürümle değişmiş olabilir.
Şunu çalıştırıp gerçek işaretleri gör, `train_on_responses_only` çağrısını ona göre düzelt:

```python
print(repr(tokenizer.apply_chat_template(
    [{'role':'user','content':'X'},{'role':'assistant','content':'Y'}],
    tokenize=False)))
```

**Kurulumda sürüm çakışması.** Ağustos 2026'da çalışan pinli set:

```python
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth_zoo bitsandbytes accelerate xformers==0.0.34 peft trl triton unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
```

**OOM.** `BATCH=1`, `ACCUM=32`, `MAXLEN=1536`; hâlâ olmuyorsa `unsloth/gemma-4-E2B-it`.

**JSON bozuk çıkıyor.** Adım 6'daki kırpma oranına bak. `max_new_tokens` de yetersiz olabilir
— bizim çıktıların p95'i ~1.400 token.

**eval_loss yükseliyor.** Aşırı öğrenme. `EPOCHS=1` yap ya da en iyi checkpoint'e dön.

**Metrikler zayıf ama loss iyi.** Bu veri setinde tipik sebep: model şablonu ezberledi ama
alan seçimini öğrenmedi. `r`'yi 64'e çıkar veya 31B'ye geç.

## Sonuçları paylaş

Adım 11 tablosunu kaydet. Zayıf çıkan metrik varsa veri setinde o görevi hedefleyen
örnek sayısı artırılabilir — üretici `generator/` altında ve tohumlu.
